# 03 — Build Feature Table (MLP)
# Giai đoạn 1 — Mục 1.4 — Xây dựng bảng đặc trưng 32 chiều cho MLP

- **Đầu ra**: `outputs/tables/features_mlp.parquet`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from scipy.signal import resample_poly
from common import io_utils, features_full, config as cfg
from common.features_full import build_full_feature_table

In [2]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
manifest_clean = pd.read_csv(TABLES_DIR / "manifest_clean.csv")

- Đọc cấu hình bandpass từ `bandpass_config.json` (đã chốt ở notebook 02)


In [4]:
with open(TABLES_DIR / "bandpass_config.json") as f:
    bandpass_cfg = json.load(f)
BAND_HZ = tuple(bandpass_cfg["band_hz"])
LP_CUTOFF_HZ = bandpass_cfg["lp_cutoff_hz"]

- Ánh xạ tần số lấy mẫu trực tiếp từ manifest để không hardcode

In [5]:
fs_map = dict(zip(manifest_clean['file_path'], manifest_clean['fs']))

def load_de_signal_fixed_fs(file_path, target_fs=12000):
    x = io_utils.load_de_signal(Path(file_path))
    fs = fs_map.get(str(file_path), target_fs)
    if fs != target_fs:
        x = resample_poly(x, target_fs, fs)
    return x

- Xây dựng bảng đặc trưng (Chốt 32 chiều tối ưu cho Edge AI/TinyML)

In [6]:
try:
    feature_df_mlp = features_full.build_full_feature_table(
        manifest_clean,
        band_hz=BAND_HZ,
        lp_cutoff=LP_CUTOFF_HZ,
        load_de_signal_fn=load_de_signal_fixed_fs,
    )
except TypeError:
    feature_df_mlp = features_full.build_full_feature_table(
        manifest_clean,
        band_hz=BAND_HZ,
        load_de_signal_fn=load_de_signal_fixed_fs,
    )

feature_df_mlp.to_parquet(TABLES_DIR / "features_mlp.parquet")
print(f"Đã lưu bảng đặc trưng MLP (32 chiều): {len(feature_df_mlp)} dòng")
display(feature_df_mlp.head())

Đã lưu bảng đặc trưng MLP (32 chiều): 40 dòng


,file_id,label,load_hp,fault_diameter_mils,time_mean,time_std,time_rms,time_peak,time_kurtosis,time_skewness,...,order_BSF_h3,envelope_BPFO_h1,envelope_BPFO_h2,envelope_BPFO_h3,envelope_BPFI_h1,envelope_BPFI_h2,envelope_BPFI_h3,envelope_BSF_h1,envelope_BSF_h2,envelope_BSF_h3
0,..\..\data\clean\118_0.mat,B,0,7.0,0.012607,0.138662,0.139234,0.607020,-0.015284,-0.008854,...,0.000116,0.001937,0.001669,0.000626,0.001918,0.000626,0.000068,0.002541,0.000508,0.000524
1,..\..\data\clean\119_1.mat,B,1,7.0,0.003892,0.139014,0.139068,0.659649,-0.036246,0.007453,...,0.000062,0.002364,0.001592,0.000743,0.002512,0.000743,0.000066,0.001227,0.000330,0.000505
2,..\..\data\clean\120_2.mat,B,2,7.0,0.004564,0.147181,0.147252,0.604584,-0.168587,0.027124,...,0.000083,0.003196,0.000954,0.000481,0.003518,0.000481,0.000060,0.001992,0.000387,0.000382
3,..\..\data\clean\121_3.mat,B,3,7.0,0.004200,0.153577,0.153635,0.720562,-0.110270,0.020412,...,0.000101,0.000889,0.001383,0.000547,0.002743,0.000094,0.000018,0.001384,0.000983,0.000214
4,..\..\data\clean\185_0.mat,B,0,14.0,0.004688,0.152642,0.152714,2.278153,14.769217,0.225132,...,0.000476,0.001030,0.000458,0.000174,0.000582,0.000174,0.000048,0.001382,0.001156,0.000465
